# InceptionV1 dataset examples — completing the triangulation (Kaggle GPU)

Feature visualization showed what a neuron *would* maximally fire on (synthesized from
noise). This notebook adds the other half: the **real photos that actually fire it most**.
When the synthesized image and the real top-activating images agree, you can trust the
label for a feature. That agreement is the core method of mechanistic interpretability.

For each chosen channel we show: **synthesized feature  |  top-6 real images**.

**Self-contained — no dataset upload needed.** It downloads Imagenette (a small 10-class
natural-image subset of ImageNet, ~99 MB) in-notebook. Just:
- Settings → **Accelerator: GPU T4**
- Settings → **Internet: On**
- **Run All**.

Output: `/kaggle/working/dataset_examples/` (one triangulation figure per channel + a
combined montage), zipped to `dataset_examples.zip` for download.

In [ ]:
# --- download a small natural-image set (Imagenette val, ~3.9k images) ---
import os, urllib.request, tarfile
URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'
if not os.path.exists('/kaggle/working/imagenette2-160'):
    print('downloading imagenette...')
    urllib.request.urlretrieve(URL, '/kaggle/working/imagenette.tgz')
    with tarfile.open('/kaggle/working/imagenette.tgz') as t:
        t.extractall('/kaggle/working')
    print('done')
IMG_ROOT = '/kaggle/working/imagenette2-160/val'
paths = []
for r, _, fs in os.walk(IMG_ROOT):
    paths += [os.path.join(r, f) for f in fs if f.endswith('.JPEG')]
print(len(paths), 'images')

In [ ]:
import math, time, shutil
import numpy as np
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1).to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)
modules = dict(model.named_modules())
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

# --- CONFIG: which units to characterise ---
LAYER = 'inception4e'
CHANNELS = [831, 525, 453, 377, 302, 151]   # set your own; 831 = the 'coil' unit we traced
print('device:', device)

In [ ]:
# --- pass every real image through, record each channel's spatial-max activation ---
prep = transforms.Compose([transforms.Resize(160), transforms.CenterCrop(160), transforms.ToTensor()])
class ImgDS(Dataset):
    def __init__(s, paths): s.paths = paths
    def __len__(s): return len(s.paths)
    def __getitem__(s, i):
        return prep(Image.open(s.paths[i]).convert('RGB')), i
loader = DataLoader(ImgDS(paths), batch_size=64, num_workers=2)

cap = {}
h = modules[LAYER].register_forward_hook(lambda m, i, o: cap.__setitem__('a', o.detach()))
maxact = None
t0 = time.time()
with torch.no_grad():
    for xb, idx in loader:
        model((xb.to(device) - MEAN) / STD)
        vmax = cap['a'].amax(dim=(2, 3)).cpu().numpy()    # [B, C]
        if maxact is None:
            maxact = np.zeros((len(paths), vmax.shape[1]), dtype=np.float32)
        maxact[idx.numpy()] = vmax
h.remove()
print(f'scored {len(paths)} images x {maxact.shape[1]} channels in {time.time()-t0:.0f}s')

In [ ]:
# --- synthesized feature (lucid recipe), so we can show synth vs real side by side ---
COLOR_CORR = torch.tensor([[0.26, 0.09, 0.02], [0.27, 0.0, -0.05], [0.27, -0.09, 0.03]], device=device)
COLOR_CORR = COLOR_CORR / torch.linalg.norm(COLOR_CORR, dim=1).max()
def _freqs(s):
    fy = np.fft.fftfreq(s)[:, None]; fx = np.fft.rfftfreq(s)[None, :]; return np.sqrt(fy ** 2 + fx ** 2)
def feature_viz(layer, ch, size=224, steps=384, lr=0.05):
    f = _freqs(size); sc = torch.tensor((1 / np.maximum(f, 1 / size)) * math.sqrt(size * size), dtype=torch.float32, device=device)[None]
    sp = (torch.randn(3, *f.shape, 2, device=device) * 0.01).requires_grad_(True)
    opt = torch.optim.Adam([sp], lr=lr); cap2 = {}
    hh = modules[layer].register_forward_hook(lambda m, i, o: cap2.__setitem__('a', o))
    def img():
        x = sp * sc[..., None]; c = torch.complex(x[..., 0], x[..., 1])
        im = torch.fft.irfft2(c, s=(size, size)) / 4.0
        return torch.sigmoid(torch.einsum('chw,dc->dhw', im, COLOR_CORR))[None]
    for _ in range(steps):
        opt.zero_grad(); im = F.pad(img(), [12] * 4, mode='reflect')
        im = TF.affine(im, angle=float(torch.empty(1).uniform_(-10, 10)), translate=[int(torch.randint(-8, 9, (1,))), int(torch.randint(-8, 9, (1,)))], scale=float(torch.empty(1).uniform_(0.9, 1.1)), shear=[0.0, 0.0], interpolation=TF.InterpolationMode.BILINEAR)
        im = im[:, :, 12:-12, 12:-12]; model((im - MEAN) / STD)
        (-cap2['a'][0, ch].mean()).backward(); opt.step()
    hh.remove()
    return np.clip(img()[0].permute(1, 2, 0).detach().cpu().numpy(), 0, 1)
print('feature_viz ready')

In [ ]:
# --- triangulation figure per channel: synth feature | top-6 real images ---
OUT = '/kaggle/working/dataset_examples'; os.makedirs(OUT, exist_ok=True)
K = 6
fig, axes = plt.subplots(len(CHANNELS), K + 1, figsize=(2.0 * (K + 1), 2.0 * len(CHANNELS)))
axes = np.atleast_2d(axes)
for r, ch in enumerate(CHANNELS):
    synth = feature_viz(LAYER, ch)
    axes[r, 0].imshow(synth); axes[r, 0].set_title(f'{LAYER} ch{ch}\n(synth)', fontsize=8)
    top = np.argsort(maxact[:, ch])[::-1][:K]
    for c, idx in enumerate(top):
        im = prep(Image.open(paths[idx]).convert('RGB')).permute(1, 2, 0).numpy()
        axes[r, c + 1].imshow(im); axes[r, c + 1].set_title(f'{maxact[idx, ch]:.1f}', fontsize=8)
    for c in range(K + 1):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    plt.imsave(f'{OUT}/{LAYER}_ch{ch:04d}_synth.png', synth)
fig.suptitle(f'{LAYER}: synthesized feature (left) vs real top-activating images', fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.98)); fig.savefig(f'{OUT}/triangulation_{LAYER}.png', dpi=150); plt.show()

In [ ]:
shutil.make_archive('/kaggle/working/dataset_examples', 'zip', OUT)
print('Done. Right sidebar -> Output -> download dataset_examples.zip,')
print('then drop it on your Desktop and tell Claude to import it.')